# Reasoning Engine Demo (LangChain + LangGraph)
This notebook is a **single-file demo** to introduce:

- A **reasoning engine** as an explicit control loop
- **Tools** with validated inputs
- A **LangGraph** state machine (model → tools → reflection → stop)
- A simple **reflection rubric** and retry policy

## How to run
1) Create a virtualenv and install:
```bash
pip install -U langchain langgraph langchain-openai pydantic
```

2) Set your model provider credentials (example for OpenAI):
```bash
export OPENAI_API_KEY="..."
```

3) Run the notebook and execute cells top to bottom.

> Notes:
> - The notebook uses `ChatOpenAI` by default, but you can swap in any LangChain-compatible chat model.
> - The code is intentionally minimal and readable.

In [ ]:
pip install -U langchain langgraph langchain-openai pydantic

In [19]:
from google.colab import userdata
import os

# Option A (recommended): set OPENAI_API_KEY in your environment before running.
# Option B: uncomment and paste your key (not recommended for shared notebooks).
# os.environ["OPENAI_API_KEY"] = "YOUR_KEY_HERE"

os.environ["OPENAI_API_KEY"] = userdata.get("gpt_chat_key")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set. Set it before running."

print("API_KEYS detected ✅")

API_KEYS detected ✅


## What this cell does (and why)

This cell sets up the **building blocks** for the entire notebook.

### Key ideas
- We import **typing primitives** (`TypedDict`, `Literal`, etc.) so the agent state and tool I/O are explicit and machine-checkable.
- We import **Pydantic** to validate tool inputs. This matters because tool calls come from an LLM and must be treated as untrusted.
- We import LangChain **messages** (`HumanMessage`, `AIMessage`, `ToolMessage`) because the agent’s working memory is a message log.
- We import `StructuredTool` to define tools with schemas and predictable invocation.
- We import LangGraph (`StateGraph`, `END`) to build a state machine for the reasoning engine.

### Architectural takeaway
We are building a **runtime**:
- shared mutable state
- nodes that transform state
- edges that route execution

In [20]:
# --- Imports ---
from __future__ import annotations

from typing import Any, Dict, List, Literal, Optional, TypedDict, Union
from dataclasses import dataclass
import os
import json
import time

from pydantic import BaseModel, Field, ValidationError

# LangChain / LangGraph
from langchain_core.tools import StructuredTool
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
    SystemMessage,
)

# Chat model (default: OpenAI). Swap this to your preferred provider.
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, END

## 1) Define tools (the action surface)
We define **two tools**:

1) `search_docs(query)` simulates knowledge retrieval (replace with real search later)
2) `calc(expression)` evaluates a safe arithmetic expression

Tools return **structured results** to reduce ambiguity and hallucination risk.

## What this cell does (and why)

This cell defines the agent’s **tools** (its action surface).

### Tool 1: `search_docs`
- Uses a tiny in-memory corpus as a stand-in for real retrieval.
- Returns structured output (`ok`, `hits`, etc.) so the model can reason over results reliably.

### Tool 2: `calc`
- A constrained calculator demonstrating safe tool design.
- Restricts characters and disables builtins to reduce abuse.

### Why `StructuredTool` + `args_schema`
- Inputs are validated (Pydantic), which prevents malformed tool calls from breaking your run.
- The model gets a clear schema, improving tool-call accuracy.

### Architectural takeaway
Tools are **contracts with reality**: validated, observable, deterministic boundaries.

In [21]:
# --- Tool 1: "search" over a tiny in-memory corpus (stand-in for real retrieval) ---

CORPUS = [
    {
        "id": "doc-1",
        "title": "Incident Playbook: 502/503 errors",
        "text": "Common causes: overloaded upstream, bad deploy, DNS issues, rate limiting. Check recent deploys and upstream health.",
    },
    {
        "id": "doc-2",
        "title": "Runbook: Database connection pool exhaustion",
        "text": "Symptoms: timeouts, increased latency. Actions: increase pool size, check leaks, scale DB, add backpressure.",
    },
    {
        "id": "doc-3",
        "title": "Observability: What to check first",
        "text": "Start with: (1) user impact, (2) recent changes, (3) error budget burn, (4) key dashboards, (5) logs and traces.",
    },
]

class SearchInput(BaseModel):
    query: str = Field(..., min_length=2, description="What you want to find in the docs")

def search_docs(query: str) -> Dict[str, Any]:
    q = query.lower()
    hits = []
    for d in CORPUS:
        hay = (d["title"] + " " + d["text"]).lower()
        score = 0
        for token in q.split():
            if token in hay:
                score += 1
        if score > 0:
            hits.append({"id": d["id"], "title": d["title"], "score": score, "snippet": d["text"][:160] + "..."})
    hits.sort(key=lambda x: x["score"], reverse=True)
    return {"ok": True, "query": query, "hits": hits[:3]}

search_tool = StructuredTool.from_function(
    func=lambda query: search_docs(query),
    name="search_docs",
    description="Search a small ops/runbook corpus for relevant information. Input: {query: string}. Output: {hits: [...]}.",
    args_schema=SearchInput,
)

# --- Tool 2: safe calculator (no eval on arbitrary code) ---

class CalcInput(BaseModel):
    expression: str = Field(..., min_length=1, description="A simple arithmetic expression, e.g. '2*(3+4)'")

ALLOWED = set("0123456789+-*/(). %")

def safe_calc(expression: str) -> Dict[str, Any]:
    expr = expression.strip()
    if any(ch not in ALLOWED for ch in expr):
        return {"ok": False, "error": "Expression contains invalid characters."}
    # Very small safety: disallow double-underscore and letters
    if any(ch.isalpha() for ch in expr) or "__" in expr:
        return {"ok": False, "error": "Expression contains invalid tokens."}
    try:
        value = eval(expr, {"__builtins__": {}}, {})
        return {"ok": True, "expression": expr, "value": value}
    except Exception as e:
        return {"ok": False, "error": f"Failed to compute expression: {e}"}

calc_tool = StructuredTool.from_function(
    func=lambda expression: safe_calc(expression),
    name="calc",
    description="Compute a basic arithmetic expression safely. Input: {expression: string}. Output: {value: number} or {error: string}.",
    args_schema=CalcInput,
)

TOOLS = [search_tool, calc_tool]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

## 2) Define the agent state
This is the **explicit runtime state** of the reasoning engine.

## What this cell does (and why)

This cell defines the **shared runtime state** for the graph.

### What state contains
- `messages`: the full trace (human, AI, tool outputs)
- `steps` / `max_steps`: step budget
- `attempts` / `max_attempts`: reflection retry budget
- `last_tool_result`: last observation (handy for debugging)
- `done`: termination flag

### Why `TypedDict`
It provides a readable contract and reduces state drift as you add nodes.

### Architectural takeaway
State is where the reasoning engine “remembers” what happened and enforces policies.

In [22]:
class AgentState(TypedDict):
    messages: List[BaseMessage]
    steps: int
    max_steps: int
    last_tool_result: Optional[Dict[str, Any]]
    done: bool
    attempts: int
    max_attempts: int

## 3) Model setup
We use `ChatOpenAI` by default. Replace with any other provider later.

Tip for teaching:
- Show where the model is "plugged in"
- Everything else is architecture: tools, state, policies, loops

## What this cell does (and why)

This cell configures the **LLM dependency** and enables tool calling.

### `make_model()`
- Uses low temperature for stable behavior in demos.
- Swap the provider here without changing the rest of the architecture.

### `bind_tools(TOOLS)`
- Exposes tool schemas to the model.
- Enables structured `tool_calls` (no string parsing).

### Architectural takeaway
The model proposes actions; the runtime executes them.

In [23]:
def make_model() -> ChatOpenAI:
    # You can swap model name here.
    # Keep temperature low for demo reliability.
    return ChatOpenAI(model="gpt-4o-mini", temperature=0)

model = make_model()

# Bind tools to enable tool calling (the model can now emit tool calls).
model_with_tools = model.bind_tools(TOOLS)

## 4) Node 1: Model step (reasoning)
This node asks the model: *"Given the conversation so far, what should we do next?"*

Outcomes:
- The model returns a normal AI message (final answer or intermediate reasoning)
- Or it includes **tool calls** that LangGraph will route to our tool executor node

## What this cell does (and why)

This cell defines `model_node`, the **decision step** of the reasoning engine.

### Responsibilities
1. Increment the step counter.
2. Build the prompt: a stable system policy + accumulated messages.
3. Call the tool-enabled model.
4. Append the `AIMessage` back into `messages`.

### Architectural takeaway
The model node decides “what next”. It does not execute tools or control looping.

In [24]:
SYSTEM = SystemMessage(content=
    "You are an incident triage assistant. "
    "Be concise. If you need external info, call tools. "
    "Prefer calling search_docs for runbook guidance before guessing."
)

def model_node(state: AgentState) -> AgentState:
    if state["done"]:
        return state
    state["steps"] += 1

    # Assemble messages with a stable system prompt
    msgs = [SYSTEM] + state["messages"]

    ai: AIMessage = model_with_tools.invoke(msgs)  # may contain tool calls
    state["messages"] = state["messages"] + [ai]
    return state

## 5) Node 2: Tool executor (act)
Executes whatever tools the model requested, appends tool results as `ToolMessage`s.

## What this cell does (and why)

This cell defines `tools_node`, the **action step**.

### Responsibilities
- Read the last `AIMessage`.
- Execute any emitted `tool_calls`.
- Append results as `ToolMessage` objects (correlated with `tool_call_id`).

### Why ToolMessage
Tool-calling models expect tool outputs as tool messages so they can incorporate observations correctly.

### Architectural takeaway
Reasoning and acting are separated for reliability and debuggability.

In [25]:
def tools_node(state: AgentState) -> AgentState:
    if state["done"]:
        return state

    last = state["messages"][-1]
    if not isinstance(last, AIMessage):
        return state

    tool_calls = last.tool_calls or []
    if not tool_calls:
        return state  # nothing to do

    for call in tool_calls:
        name = call["name"]
        args = call.get("args", {})
        tool = TOOLS_BY_NAME.get(name)
        if tool is None:
            tool_result = {"ok": False, "error": f"Unknown tool: {name}"}
        else:
            try:
                tool_result = tool.invoke(args)
            except ValidationError as ve:
                tool_result = {"ok": False, "error": f"Tool input validation failed: {ve}"}
            except Exception as e:
                tool_result = {"ok": False, "error": f"Tool execution failed: {e}"}

        state["last_tool_result"] = tool_result

        # ToolMessage requires tool_call_id to correlate
        tm = ToolMessage(
            content=json.dumps(tool_result, ensure_ascii=False),
            tool_call_id=call["id"],
            name=name,
        )
        state["messages"] = state["messages"] + [tm]

    return state

## 6) Node 3: Reflection (quality control)
Reflection is a **policy layer**, not mysticism.

We ask the model to judge the current draft answer against a rubric:
- Did we use tools when needed?
- Is the answer actionable?
- Are we making unsupported claims?

If the critique is poor, we ask the model to revise.

## What this cell does (and why)

This cell implements `reflection_node`, a bounded **quality-control loop**.

### What it checks
- Did we use tools before guessing?
- Is the answer actionable?
- Are we making unsupported claims?
- Is it concise?

### Control features
- Parses a structured JSON critique.
- If revision is required, appends a revision instruction to `messages`.
- Caps retries via `max_attempts` to prevent infinite loops.

### Architectural takeaway
Reflection is a policy layer: explicit rubric + bounded retries.

In [33]:
class Critique(BaseModel):
    verdict: Literal["pass", "revise"]
    issues: List[str] = Field(default_factory=list)
    improved_answer: Optional[str] = None

REFLECT_PROMPT = SystemMessage(content=
    "You are a strict reviewer. Evaluate the assistant's latest answer. "
    "Rubric: (1) Uses tools before guessing when facts/runbooks are needed, "
    "(2) Actionable steps, (3) No unsupported claims, (4) Concise. "
    "If revision is needed, provide improved_answer."
)

def reflection_node(state: AgentState) -> AgentState:
    if state["done"]:
        return state

    # Find the most recent AI message that isn't just tool-call scaffolding
    # We'll critique the last AIMessage content.
    last_ai = None
    for m in reversed(state["messages"]):
        if isinstance(m, AIMessage):
            last_ai = m
            break
    if last_ai is None:
        return state

    # If the last AI message is still requesting tools (has tool_calls),
    # we skip reflection until after tools run and the model responds again.
    if last_ai.tool_calls:
        return state

    critique_request = [
        REFLECT_PROMPT,
        HumanMessage(content=f"Critique this answer:\n\n{last_ai.content}\n\nReturn JSON with keys: verdict, issues, improved_answer."),
    ]

    critique_msg = model.invoke(critique_request)
    raw = critique_msg.content

    # Best-effort JSON parse
    try:
        data = json.loads(raw) if isinstance(raw, str) else raw
        critique = Critique.model_validate(data)
    except Exception:
        # If parsing fails, default to pass to avoid infinite loops in demo.
        critique = Critique(verdict="pass", issues=["Critique parsing failed; skipping."])

    if critique.verdict == "pass":
        return state

    state["attempts"] += 1
    if state["attempts"] >= state["max_attempts"]:
        # Stop revising, keep best-effort answer
        state["done"] = True
        return state

    # Append a revision request to messages so the model produces a better answer next step.
    revision_instruction = HumanMessage(content=
        "Revise your last answer using this critique:\n"
        f"- Issues: {critique.issues}\n"
        "Return a revised, concise, actionable answer. If you need info, call tools."
    )
    state["messages"] = state["messages"] + [revision_instruction]
    return state

## 7) Stop conditions and routing
We route based on the last message:

- If the model emitted tool calls → go to tools
- Else if we have a final-ish answer and reflection passes or attempts exhausted → stop
- Else → loop to model

LangGraph lets us encode this explicitly.

## What this cell does (and why)

This cell defines routing: **state → next node**.

### What it encodes
- If the model asked for tools → go to `tools`.
- If the model produced an answer → go to `reflect`.
- If budget is exceeded → end.

### Architectural takeaway
Routing is where your reasoning engine policy lives. Prompts cannot replace it.

In [34]:
def should_go_to_tools(state: AgentState) -> bool:
    last = state["messages"][-1]
    return isinstance(last, AIMessage) and bool(last.tool_calls)

def should_stop(state: AgentState) -> bool:
    if state["done"]:
        return True
    if state["steps"] >= state["max_steps"]:
        return True
    # Stop if last AI message has no tool calls AND last user instruction wasn't a revision request
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and not last.tool_calls:
        return False  # let reflection decide next
    return False

def router(state: AgentState) -> str:
    if state["steps"] >= state["max_steps"]:
        return "end"
    if should_go_to_tools(state):
        return "tools"
    # If last is an AI message with no tool calls, run reflection next
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and not last.tool_calls:
        return "reflect"
    # Otherwise go back to model
    return "model"

## 8) Build the LangGraph app

## What this cell does (and why)

This cell constructs the **LangGraph state machine**.

### Steps
- Add nodes: `model`, `tools`, `reflect`
- Define entry point: `model`
- Add conditional edges from `model` via `router`
- Loop tools back to model so observations are incorporated
- Compile into `app` so we can run `app.invoke(...)`

### Architectural takeaway
This is the reasoning engine: behaviors (nodes) + policies (edges) + memory (state).

In [35]:
graph = StateGraph(AgentState)

graph.add_node("model", model_node)
graph.add_node("tools", tools_node)
graph.add_node("reflect", reflection_node)

graph.set_entry_point("model")

# Conditional edges from model
graph.add_conditional_edges(
    "model",
    router,
    {
        "tools": "tools",
        "reflect": "reflect",
        "model": "model",
        "end": END,
    },
)

# After tools, go back to model to incorporate observations
graph.add_edge("tools", "model")

# After reflection, go back to model (if revised) or end (if done or attempts exhausted)
def after_reflect_router(state: AgentState) -> str:
    if state["done"] or state["steps"] >= state["max_steps"]:
        return "end"
    # If reflection appended a revision request, go to model
    return "model"

graph.add_conditional_edges(
    "reflect",
    after_reflect_router,
    {"model": "model", "end": END},
)

app = graph.compile()

## 9) Run the demo: incident triage scenario
This is the single scenario you can narrate in the video.

Try different prompts:
- ambiguous
- tool-worthy
- purely arithmetic (should call calc)

## What this cell does (and why)

This cell provides a reusable `run()` helper.

### Why it exists
- Creates an initial state consistently.
- Lets you tune step and retry budgets from one place.
- Makes experiments reproducible.

### Architectural takeaway
Always formalize the initial state. It enables replay, testing, and debugging.

In [36]:
def run(prompt: str, max_steps: int = 12, max_attempts: int = 2) -> AgentState:
    init: AgentState = {
        "messages": [HumanMessage(content=prompt)],
        "steps": 0,
        "max_steps": max_steps,
        "last_tool_result": None,
        "done": False,
        "attempts": 0,
        "max_attempts": max_attempts,
    }
    out = app.invoke(init)
    return out

state = run("We are seeing lots of 503s after a deploy. What should I check first and what are the likely causes?")
print("Steps:", state["steps"], "Attempts:", state["attempts"])

Steps: 12 Attempts: 0


## What this cell does (and why)

This cell prints a human-readable trace of the run.

### Why tracing matters
You need to see:
- when tools were called
- what args were used
- what observations came back
- where the loop repeated or got stuck

### Architectural takeaway
Observability is non-negotiable for agent reliability.

In [37]:
# Pretty-print the conversation trace
def show_trace(state: AgentState) -> None:
    for i, m in enumerate(state["messages"], 1):
        role = m.type
        content = m.content
        if isinstance(m, AIMessage) and m.tool_calls:
            print(f"\n[{i}] AI (tool calls):")
            for tc in m.tool_calls:
                print("  -", tc["name"], tc.get("args", {}))
        else:
            print(f"\n[{i}] {role.upper()}:")
            print(content)

show_trace(state)


[1] HUMAN:
We are seeing lots of 503s after a deploy. What should I check first and what are the likely causes?

[2] AI (tool calls):
  - search_docs {'query': '503 errors after deploy causes and checks'}

[3] TOOL:
{"ok": true, "query": "503 errors after deploy causes and checks", "hits": [{"id": "doc-1", "title": "Incident Playbook: 502/503 errors", "score": 5, "snippet": "Common causes: overloaded upstream, bad deploy, DNS issues, rate limiting. Check recent deploys and upstream health...."}, {"id": "doc-3", "title": "Observability: What to check first", "score": 1, "snippet": "Start with: (1) user impact, (2) recent changes, (3) error budget burn, (4) key dashboards, (5) logs and traces...."}]}

[4] AI:
You should check the following first:

1. **Recent Deploys**: Verify if the latest deployment introduced any issues.
2. **Upstream Health**: Check if the upstream services are overloaded or down.
3. **DNS Issues**: Ensure that DNS resolution is functioning correctly.
4. **Rate Limi

## 10) Try a calc request (tool calling)

## What this cell does (and why)

This cell prints a human-readable trace of the run.

### Why tracing matters
You need to see:
- when tools were called
- what args were used
- what observations came back
- where the loop repeated or got stuck

### Architectural takeaway
Observability is non-negotiable for agent reliability.

In [38]:
state2 = run("Compute 2*(3+7)/5 and explain the steps briefly.")
show_trace(state2)


[1] HUMAN:
Compute 2*(3+7)/5 and explain the steps briefly.

[2] AI (tool calls):
  - calc {'expression': '2*(3+7)/5'}

[3] TOOL:
{"ok": true, "expression": "2*(3+7)/5", "value": 4.0}

[4] AI:
The computation of \( 2 \times (3 + 7) / 5 \) can be broken down into the following steps:

1. **Parentheses First**: Calculate the expression inside the parentheses: \( 3 + 7 = 10 \).
2. **Multiplication**: Multiply the result by 2: \( 2 \times 10 = 20 \).
3. **Division**: Finally, divide by 5: \( 20 / 5 = 4 \).

Thus, the final result is \( 4.0 \).

[5] AI:
The computation of \( 2 \times (3 + 7) / 5 \) can be broken down into the following steps:

1. **Parentheses First**: Calculate the expression inside the parentheses: \( 3 + 7 = 10 \).
2. **Multiplication**: Multiply the result by 2: \( 2 \times 10 = 20 \).
3. **Division**: Finally, divide by 5: \( 20 / 5 = 4 \).

Thus, the final result is \( 4.0 \).

[6] AI:
The computation of \( 2 \times (3 + 7) / 5 \) can be broken down into the followin

## 11) Important notes

- The **reasoning engine** is the graph: nodes + state + routing.
- The model is a **policy-driven decision function**, not the whole system.
- Tools are your **capabilities**, and schemas are your reliability.
- Reflection turns quality into an **engineering loop**.
- Stop conditions prevent infinite loops and runaway cost.